In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torch
import pickle
from helpers.predictors import initialize_models, add_object, track_object, render_bbox_video

USE_TAM = True
# Initialize models with TAM tracker
initialize_models(use_tam=USE_TAM)

data_collection = 'cha'  # Data collection to use 

# Configuration
data_dir = f'inputs/{data_collection}'  # Base directory for inputs
detection_dir = f'outputs/{data_collection}'  # Directory containing detection txt files
show=False  # Show the video while processing
original_resolution = (3840, 2160)
resolution = (960, 540)  # Resolution of the video in case of resampling

max_hands = 2  # Maximum number of hands to track

Device: cuda
Torch compile enabled: True
Loads checkpoint by local backend from path: checkpoints/hands/detection/cascade_rcnn_x101_64x4d_fpn_20e_onehand10k-dac19597_20201030.pth
Loads checkpoint by local backend from path: checkpoints/body/detection/rtmdet_m_8xb32-100e_coco-obj365-person-235e8209.pth
Disable torch compile due to unsupported GPU.
Models initialized with EfficientTAM tracker


In [ ]:
def read_detection_file(filepath):
    """Read detection file containing frame index and bounding boxes"""
    detections = {}
    with open(filepath, 'r') as f:
        for line in f:
            values = list(map(float, line.strip().split()))
            frame_idx = int(values[0])
            boxes = []
            # Each box has 4 coordinates
            for i in range(1, len(values), 4):
                boxes.append(values[i:i+4])

            boxes = np.array(boxes)

            if resolution != original_resolution:
                # Rescale bounding boxes to original resolution
                
                boxes[:, 0] *= resolution[0] / original_resolution[0]
                boxes[:, 1] *= resolution[1] / original_resolution[1]
                boxes[:, 2] *= resolution[0] / original_resolution[0]
                boxes[:, 3] *= resolution[1] / original_resolution[1]
            detections[frame_idx] = boxes
    return detections

def prepare_video_frames(video_path, resample_to=None):
    """Create frames directory if video file exists"""
    if not os.path.exists(video_path):
        raise FileNotFoundError(f"Video not found: {video_path}")
        
    video_name = os.path.splitext(os.path.basename(video_path))[0]
    frames_dir = os.path.join(os.path.dirname(video_path), video_name)
    
    if not os.path.exists(frames_dir):
        os.makedirs(frames_dir)
        cap = cv2.VideoCapture(video_path)
        frame_idx = 0
        
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            if resample_to is not None:
                frame = cv2.resize(frame, resample_to)
            cv2.imwrite(os.path.join(frames_dir, f"{frame_idx:05d}.jpg"), frame)
            frame_idx += 1
            
        cap.release()
        
    return frames_dir

In [ ]:
def process_video(video_name):
    """Process a single video file and track all boxes as one object"""

    def box_tracked(tracks, frame_idx, box):
        """Check if box is already tracked in the tracks"""

        box_mean_x = (box[0] + box[2]) / 2
        box_mean_y = (box[1] + box[3]) / 2
        if frame_idx not in tracks:
            return False
        
        for tracked_box in tracks[frame_idx].values():
            tracked_box = tracked_box.flatten()
            tracked_box_mean_x = (tracked_box[0] + tracked_box[2]) / 2
            tracked_box_mean_y = (tracked_box[1] + tracked_box[3]) / 2
            tracked_in_box = (tracked_box_mean_x >= box[0] and tracked_box_mean_x <= box[2] and tracked_box_mean_y >= box[1] and tracked_box_mean_y <= box[3])
            box_in_tracked = (box_mean_x >= tracked_box[0] and box_mean_x <= tracked_box[2] and box_mean_y >= tracked_box[1] and box_mean_y <= tracked_box[3])
            if tracked_in_box and box_in_tracked:
                return True
        return False

    # Get paths
    video_path = os.path.join(data_dir, f"{video_name}.MP4")
    detection_path = os.path.join(detection_dir, f"{video_name}.txt")
    
    if not os.path.exists(detection_path):
        print(f"No detection file found for {video_name}")
        return None, None
    
    # Prepare video frames
    frames_dir = prepare_video_frames(video_path, resolution)
    
    # Read detections
    detections = read_detection_file(detection_path)
    
    # Initialize tracker for all boxes under single ID
    obj_id = 0
    all_tracks = None  
    predictor = None
    inference_state = None 

    
    # Process each frame with detections
    for frame_idx, boxes in detections.items():        
        for box in boxes:
            if (all_tracks is None or not box_tracked(all_tracks, frame_idx, box)) and obj_id < max_hands:
                print(f"Frame {frame_idx}: {box}")
                print(f"Object ID: {obj_id}")
                # Initialize tracker with first frame's boxes if not done yet
                inference_state, predictor, frame_names = add_object(
                    frames_dir, 
                    input_box=box,
                    frame_idx=frame_idx, 
                    obj_id=obj_id, 
                    show=show,
                    inference_state=inference_state,
                    predictor_in=predictor)
            
                # Track the object through video
                _, track_boxes = track_object(
                    frames_dir, 
                    inference_state, 
                    predictor, 
                    frame_names, 
                    show=show, 
                    prev_bboxes=None)
                
                obj_id += 1
                all_tracks = track_boxes
    
    return all_tracks, frame_names


In [ ]:
def adjust_tracks(tracks, adj_resolution=resolution, orig_resolution=original_resolution):
    for frame_idx, frame_tracks in tracks.items():
        for obj_id, box in frame_tracks.items():
            box = box.flatten()
            box[0] *= orig_resolution[0] / adj_resolution[0]
            box[1] *= orig_resolution[1] / adj_resolution[1]
            box[2] *= orig_resolution[0] / adj_resolution[0]
            box[3] *= orig_resolution[1] / adj_resolution[1]
            tracks[frame_idx][obj_id] = box

In [ ]:
# Process a single video
# video_name = "gopro11_synced_cut"  # Change this to process different videos
# tracks, frame_names = process_video(video_name)
# render_bbox_video(f"outputs/{data_collection}/{video_name}_tracked", os.path.join(data_dir, video_name), tracks, frame_names)

In [ ]:
# # Process videos
for video_name in os.listdir(data_dir):
    if video_name.endswith('.MP4'):
        video_name = os.path.splitext(video_name)[0]
        print(f"Processing {video_name}...")
        tracks, frame_names = process_video(video_name)
        if tracks is not None:
            print(f"Processed {video_name}: {len(tracks)} frames")

            # Render video with bounding boxes
            render_bbox_video(f"outputs/{data_collection}/{video_name}_tracked", os.path.join(data_dir, video_name), tracks, frame_names)

            with open(f"outputs/{data_collection}/tracked_bboxes_{video_name}.pkl", 'wb+') as f:
                tracks = adjust_tracks(tracks)
                pickle.dump(tracks, f)
        else:
            print(f"Failed to process {video_name}")

    


Processing gopro10_synced...
No detection file found for gopro10_synced
Failed to process gopro10_synced
Processing gopro10_synced_cut...
Frame 17: [473.14239502 146.21948242 524.72088623 201.33529663]
Object ID: 0


propagate in video:   3%|▎         | 8/283 [00:05<03:24,  1.34it/s]


KeyboardInterrupt: 